In [8]:
from utils import *
from concurrent.futures import ProcessPoolExecutor
import concurrent
%matplotlib inline

In [9]:
EJ = 3
EC = 0.6
EL = 0.13
Er = 7.2622522
g_strength = 0.3

qubit_level = 30
osc_level = 30


qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=110,truncated_dim=qubit_level)
osc = scqubits.Oscillator(E_osc=Er,truncated_dim=osc_level)
hilbertspace = scqubits.HilbertSpace([qbt, osc])
hilbertspace.add_interaction(g_strength=g_strength,op1=qbt.n_operator,op2=osc.creation_operator,add_hc=True)
hilbertspace.generate_lookup()
product_to_dressed = generate_single_mapping(hilbertspace.hamiltonian())
energies=  hilbertspace.eigenvals(qubit_level*osc_level)

products_to_keep = []

for ql in [0]:
    for ol in range(30):
        products_to_keep.append([ql,ol])
for ql in range(1,3):
    for ol in range(10):
        products_to_keep.append([ql,ol])


def truncate_function(qobj):
    return truncate_custom(qobj, products_to_keep, product_to_dressed)

def pad_back_function(qobj):
    return pad_back_custom(qobj, products_to_keep, product_to_dressed)

a = hilbertspace.op_in_dressed_eigenbasis(op=osc.annihilation_operator)
a = qutip.Qobj(a[:, :])
a_trunc = truncate_function(a)

(evals,) = hilbertspace["evals"]
diag_dressed_hamiltonian = (
        2 * np.pi * qutip.Qobj(np.diag(evals),
        dims=[hilbertspace.subsystem_dims] * 2)
)
diag_dressed_hamiltonian = qutip.Qobj(diag_dressed_hamiltonian[:, :])
diag_dressed_hamiltonian = truncate_function(diag_dressed_hamiltonian)

w_d = transition_frequency(hilbertspace,product_to_dressed[(0,0)],product_to_dressed[(0,1)] ) 

tot_time =478
tlist = np.linspace(0, tot_time, tot_time)

amp = 0.004
kappa = 0.001
decay_term = np.sqrt(kappa) * a_trunc

def square_cos(t,*args):
    cos = np.cos(w_d * 2*np.pi * t)
    return  2*np.pi * amp * cos


H_with_drive = [
    diag_dressed_hamiltonian,
    [a_trunc+a_trunc.dag(), square_cos],
]


# On reference states:

In [10]:
state_0_dressed = qutip.basis(hilbertspace.dimension, product_to_dressed[(1,0)])
state_1_dressed = qutip.basis(hilbertspace.dimension, product_to_dressed[(2,0)])
state_plus_dressed = (state_0_dressed  +  state_1_dressed).unit()
state_minus_dressed = (state_0_dressed - state_1_dressed).unit()
initial_states  = [state_0_dressed,state_1_dressed,state_plus_dressed,state_minus_dressed ]

ntraj = 400

# tomo_results = [None] * 4
    
# for i in range(4):
#     print(f'doing qubit initial state {i}')
#     tomo_results[i] = qutip.mcsolve(psi0=truncate_function(initial_states[i]), 
#                                H=H_with_drive,
#                                progress_bar = qutip.ui.progressbar.EnhancedTextProgressBar(),
#                                tlist=tlist, 
#                                c_ops = [decay_term],
#                                ntraj = ntraj,
#                                options=qutip.Options(store_states=True)
#                                )

# import pickle
# with open('mcsolve_tomo_results_kappa1e-3.pkl', 'wb') as file:
#     pickle.dump(tomo_results, file)


# result.states is a 2-d array of shape (n_traj, len(tlist)), each element is a ket,

## we first do averaging over the ntraj kets

In [11]:
with open('mcsolve_tomo_results_kappa1e-3.pkl', 'rb') as file:
    tomo_results = pickle.load(file)
ntraj = 400
dm_tomo_results_list = []
for result in tomo_results:
    new_result = qutip.solver.Result()
    new_result.times = result.times

    states_array = np.array([[state.full() for state in traj] for traj in result.states])
    summed_dm_array = np.einsum('ntrc,ntij->tri', states_array, states_array.conj()) # n is traj index, t is time index, r is row index, c is column index, i and j are the row and column index of the conjugated ket
    averaged_dm_array = summed_dm_array/ntraj
    new_result.states = [qutip.Qobj(dm) for dm in averaged_dm_array]

    dm_tomo_results_list.append(new_result)


In [12]:
def plot_heatmap(result, time_index, product_to_dressed, qubit_levels, oscillator_levels):
    if hasattr(result, 'states'):
        dm = result.states[time_index]
    elif hasattr(result, 'y'):
        dm = result.y[time_index]

    dm = pad_back_function(dm)
    grid = np.zeros(( qubit_levels,oscillator_levels))

    for qubit_level in range(qubit_levels):
        for oscillator_level in range(oscillator_levels):
            product_state = (qubit_level, oscillator_level)
            dressed_state = product_to_dressed[product_state]
            if dressed_state < dm.dims[0][0]:
                # Create a basis state corresponding to the dressed state
                basis_state = qutip.basis(dm.dims[0][0], dressed_state)
                # Calculate the expectation value
                expectation_value = qutip.expect(basis_state * basis_state.dag(), dm)
            else:
                expectation_value = 0
            grid[ qubit_level,oscillator_level] = expectation_value
    grid[grid < 1e-5] = 1e-5
    plt.imshow(grid, cmap='viridis', origin='lower')
    plt.colorbar(label='Expectation Value')
    plt.xlabel('Oscillator Level')
    plt.ylabel('Qubit Level')
    plt.title(f'Expectation Values at t = {result.times[time_index]}')
    plt.show()

def interactive_heatmap(result, product_to_dressed, qubit_levels, oscillator_levels):
    if hasattr(result, 'times'):
        times = result.times
    elif hasattr(result, 't'):
        times = result.t
    time_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(times) - 1,
        step=1,
        description='Time Index:',
        continuous_update=False
    )
    
    widgets.interact(lambda time_index: plot_heatmap(result, time_index, product_to_dressed, qubit_levels, oscillator_levels),
                     time_index=time_slider)
    

In [14]:
interactive_heatmap(dm_tomo_results_list[0], product_to_dressed, qubit_levels=qubit_level, oscillator_levels=osc_level)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Time Index:', max=477), Output(…

In [25]:
ql = 1
dm = dm_tomo_results_list[ql].states[226]
dm = pad_back_function(dm)

pop_sum = 0
for qubit_level in range(qubit_level):
    # if qubit_level != ql:
    for oscillator_level in range(osc_level):
        product_state = (qubit_level, oscillator_level)
        dressed_state = product_to_dressed[product_state]
        basis_state = qutip.basis(dm.dims[0][0], dressed_state)
        expectation_value = qutip.expect(basis_state * basis_state.dag(), dm)
        pop_sum += expectation_value

pop_sum

0

In [26]:
dm

Quantum object: dims = [[900], [900]], shape = (900, 900), type = oper, isherm = True
Qobj data =
[[ 4.80182548e-10+0.00000000e+00j  3.51135367e-11+4.91573115e-11j
  -1.23459307e-08+3.38104563e-08j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 3.51135367e-11-4.91573115e-11j  2.06877287e-09+0.00000000e+00j
  -3.95865653e-08+3.53946346e-07j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [-1.23459307e-08-3.38104563e-08j -3.95865653e-08-3.53946346e-07j
   4.25781114e-02+0.00000000e+00j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 ...
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0